In [5]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../Data/Raw")

# 1. Load
tickets = pd.read_csv(DATA_DIR / "Customer_Service.csv")

tickets["Open_Date"] = pd.to_datetime(tickets["Open_Date"], errors="coerce")

tickets["Low_CSAT"] = tickets["CSAT_1_to_5"] <= 3
tickets["Unresolved"] = tickets["Resolved"] == "No"
tickets["Return_Exchange"] = tickets["Issue_Type"].isin([
    "Return request",
    "Exchange request"
])
tickets["Wholesale_Inquiry"] = tickets["Issue_Type"] == "Wholesale inquiry"

# 5. Sanity check — must match Step 1 exactly
print(tickets.shape)                    
print(tickets["Low_CSAT"].mean().round(4))    
print(tickets["Unresolved"].mean().round(4)) 

(6500, 13)
0.2617
0.0492


In [6]:
tickets.groupby("Resolved")["Low_CSAT"].mean().round(3)

Resolved
No     0.822
Yes    0.233
Name: Low_CSAT, dtype: float64

In [7]:
tickets["Response_Bucket"] = pd.cut(
    tickets["First_Response_Minutes"],
    bins=[0, 10, 20, 30, 60, 100],
    labels=["0-10", "10-20", "20-30", "30-60", "60+"]
)

tickets.groupby("Response_Bucket", observed=True)["Low_CSAT"].mean().round(3)

Response_Bucket
0-10     0.267
10-20    0.258
20-30    0.244
30-60    0.246
60+      0.364
Name: Low_CSAT, dtype: float64

In [16]:
tickets["Resolution_Bucket"] = pd.cut(
    tickets["Resolution_Hours"],
    bins=[0, 6, 12, 24, 48, 100],
    labels=["0-6", "6-12", "12-24", "24-48", "48+"]
)

tickets.groupby("Resolution_Bucket", observed=True)["Low_CSAT"].mean().round(3)

Resolution_Bucket
0-6      0.229
6-12     0.238
12-24    0.238
24-48    0.264
48+      0.797
Name: Low_CSAT, dtype: float64

In [18]:
tickets.groupby("Channel")["Low_CSAT"].mean().round(3)


Channel
Instagram    0.267
Phone        0.247
Website      0.261
WhatsApp     0.272
Name: Low_CSAT, dtype: float64

In [19]:

tickets.groupby("Issue_Type")["Low_CSAT"].mean().round(3).sort_values(ascending=False)

Issue_Type
Product quality             0.275
Exchange request            0.273
General product question    0.270
Size guidance               0.262
Order status                0.258
Payment                     0.256
Return request              0.253
Wholesale inquiry           0.237
Name: Low_CSAT, dtype: float64

In [20]:
w = tickets[tickets["Issue_Type"] == "Wholesale inquiry"]

wholesale_profile = {
    "wholesale_low_csat_rate": round(w["Low_CSAT"].mean(), 3),
    "wholesale_unresolved_rate": round(w["Unresolved"].mean(), 3),
    "wholesale_avg_first_response": round(w["First_Response_Minutes"].mean(), 2),
    "wholesale_avg_resolution_hours": round(w["Resolution_Hours"].mean(), 2),
    "overall_low_csat_rate": round(tickets["Low_CSAT"].mean(), 3),
    "overall_unresolved_rate": round(tickets["Unresolved"].mean(), 3),
    "overall_avg_first_response": round(tickets["First_Response_Minutes"].mean(), 2),
    "overall_avg_resolution_hours": round(tickets["Resolution_Hours"].mean(), 2)
}

pd.Series(wholesale_profile).to_frame("Value")

,Value
wholesale_low_csat_rate,0.237
wholesale_unresolved_rate,0.033
wholesale_avg_first_response,11.640
wholesale_avg_resolution_hours,16.940
overall_low_csat_rate,0.262
overall_unresolved_rate,0.049
overall_avg_first_response,11.510
overall_avg_resolution_hours,17.440


In [21]:
tickets.groupby("Unresolved").agg(
    count=("Ticket_ID", "count"),
    avg_first_response=("First_Response_Minutes", "mean"),
    avg_resolution_hours=("Resolution_Hours", "mean"),
    avg_csat=("CSAT_1_to_5", "mean")
).round(2)

,count,avg_first_response,avg_resolution_hours,avg_csat
Unresolved,,,,
False,6180,11.51,15.08,4.08
True,320,11.40,63.12,2.32
